# Fixed Effect Panel Regression: Provincial Poverty Determinants in Indonesia

This notebook estimates panel regression models to analyze relationships between socioeconomic indicators and provincial poverty rates in Indonesia. The selected inferential model is Fixed Effects with robust standard errors.

## Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from linearmodels.panel import PooledOLS, PanelOLS, RandomEffects

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'data_bps_datmin.csv'
OUTPUT_DIR = Path('panel_output')
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13

print('All libraries were imported successfully.')
import linearmodels
print(f'linearmodels version: {linearmodels.__version__}')
print(f'statsmodels version: {sm.__version__}')


## Data Loading and Exploration

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = [
    'province', 'year', 'aps_1315', 'aps_1618', 'aps_1924',
    'tpt_feb', 'tpt_aug', 'tpak_feb', 'tpak_aug',
    'poverty_line_march', 'poverty_line_september',
    'poor_population_march', 'poor_population_september',
    'poverty_pct_march', 'poverty_pct_september',
    'hdi', 'mean_years_schooling', 'expected_years_schooling'
]

print('Dataset Information')
print(f'Shape: {df_raw.shape}')
print(f'Provinces: {df_raw["province"].nunique()}')
print(f'Years: {sorted(df_raw["year"].unique())}')
print()
df_raw.head()


In [ ]:
print('Descriptive Statistics')
print(df_raw.describe().to_string())


## Data Preprocessing and Feature Engineering

In [ ]:
df = df_raw.copy()

df.columns = [
    'province', 'year', 'aps_1315', 'aps_1618', 'aps_1924',
    'tpt_feb', 'tpt_aug', 'tpak_feb', 'tpak_aug',
    'poverty_line_march', 'poverty_line_september',
    'poor_population_march', 'poor_population_september',
    'poverty_pct_march', 'poverty_pct_september',
    'hdi', 'mean_years_schooling', 'expected_years_schooling'
]

df['poverty_rate'] = (df['poverty_pct_march'] + df['poverty_pct_september']) / 2

df['tpt'] = (df['tpt_feb'] + df['tpt_aug']) / 2
df['tpak'] = (df['tpak_feb'] + df['tpak_aug']) / 2
df['poverty_line'] = (df['poverty_line_march'] + df['poverty_line_september']) / 2

df['log_poverty_line'] = np.log(df['poverty_line'])

df['province'] = df['province'].str.strip().str.upper()

print('Missing Values')
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print('No missing values.')

print(f'\nFinal shape: {df.shape}')
print()
print('Variables used:')
print('  poverty_rate : average poverty rate (March + September) / 2')
print('  ipm          : Human Development Index')
print('  tpt          : average open unemployment rate')
print('  tpak         : average labor force participation rate')
print('  aps_1315     : school participation rate ages 13-15')
print('  rls          : Mean Years of Schooling')
print('  hls          : Expected Years of Schooling')
print('  log_poverty_line       : natural log of average poverty line')


In [ ]:
vars_corr = ['poverty_rate', 'hdi', 'tpt', 'tpak', 'aps_1315', 'mean_years_schooling', 'expected_years_schooling', 'log_poverty_line']
corr_matrix = df[vars_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            mask=mask, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Panel Variable Correlation Matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plot_panel_correlation.png', bbox_inches='tight')
plt.show()

print('\nCorrelation with Poverty Rate')
corr_with_y = corr_matrix['poverty_rate'].drop('poverty_rate').sort_values()
print(corr_with_y.round(4).to_string())


In [ ]:
vars_plot = ['poverty_rate', 'hdi', 'tpt', 'tpak', 'mean_years_schooling']
var_labels = {
    'poverty_rate': 'Poverty (%)',
    'hdi': 'HDI',
    'tpt': 'TPT (%)',
    'tpak': 'TPAK (%)',
    'mean_years_schooling': 'Mean Years of Schooling (years)'
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

years = sorted(df['year'].unique())

for idx, var in enumerate(vars_plot):
    ax = axes[idx]
    grouped = df.groupby('year')[var].agg(['mean', 'std', 'min', 'max'])
    ax.plot(grouped.index, grouped['mean'], 'bo-', linewidth=2, markersize=7, label='Average')
    ax.fill_between(grouped.index,
                    grouped['mean'] - grouped['std'],
                    grouped['mean'] + grouped['std'],
                    alpha=0.2, color='blue', label='±1 SD')
    ax.fill_between(grouped.index, grouped['min'], grouped['max'],
                    alpha=0.08, color='blue', label='Min-Max')
    ax.set_xlabel('Year')
    ax.set_ylabel(var_labels[var])
    ax.set_title(f'Trend of {var_labels[var]}')
    ax.set_xticks(years)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.4)

    for t in years:
        val = grouped.loc[t, 'mean']
        ax.annotate(f'{val:.2f}', xy=(t, val),
                    xytext=(0, 8), textcoords='offset points',
                    fontsize=8, ha='center')

ax6 = axes[5]
for prov in df['province'].unique():
    sub = df[df['province'] == prov].sort_values('year')
    ax6.plot(sub['year'], sub['poverty_rate'],
             alpha=0.35, linewidth=0.9, color='steelblue')

mean_trend = df.groupby('year')['poverty_rate'].mean()
ax6.plot(mean_trend.index, mean_trend.values, 'r-o', linewidth=2.5,
         markersize=7, label='National average', zorder=5)
ax6.set_xlabel('Year')
ax6.set_ylabel('Poverty (%)')
ax6.set_title('Provincial Poverty Trend (Spaghetti Plot)')
ax6.set_xticks(years)
ax6.legend()
ax6.grid(alpha=0.4)

plt.suptitle('BPS Panel Data Exploration 2021-2024', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plot_panel_trends.png', bbox_inches='tight')
plt.show()


In [ ]:
print('Variation Decomposition (Between vs Within)')
print(f'{"Variable":35s} {"Total SD":>10s} {"Between SD":>12s} {"Within SD":>11s} {"Within/Total":>13s}')
print('-' * 83)

for var in ['poverty_rate', 'hdi', 'tpt', 'tpak', 'mean_years_schooling', 'expected_years_schooling', 'log_poverty_line', 'aps_1315']:
    total_sd = df[var].std()

    prov_mean = df.groupby('province')[var].mean()
    between_sd = prov_mean.std()

    df_temp = df.copy()
    df_temp['prov_mean'] = df_temp.groupby('province')[var].transform('mean')
    within_sd = (df_temp[var] - df_temp['prov_mean']).std()

    ratio = within_sd / total_sd if total_sd > 0 else 0
    print(f'{var:35s} {total_sd:>10.4f} {between_sd:>12.4f} {within_sd:>11.4f} {ratio:>12.2%}')


## Panel Data Setup

The `linearmodels` package requires a MultiIndex structure with province as the entity index and year as the time index.

In [ ]:
VARS_MODEL = ['province', 'year', 'poverty_rate', 'hdi', 'tpt', 'tpak', 'aps_1315', 'mean_years_schooling', 'expected_years_schooling', 'log_poverty_line']
df_panel = df[VARS_MODEL].copy()

df_panel = df_panel.set_index(['province', 'year'])

print('Panel Data (MultiIndex)')
print(f'Shape: {df_panel.shape}')
print(f'Level 0 (Province): {df_panel.index.get_level_values(0).nunique()} unit')
print(f'Level 1 (Year): {df_panel.index.get_level_values(1).nunique()} periode')
print(f'Total observations: {len(df_panel)}')
print()

counts = df_panel.groupby(level=0).size()
print(f'Panel balanced? {counts.nunique() == 1}')
print(f'Observations by province: min={counts.min()}, max={counts.max()}, unique={counts.unique()}')
print()
print(df_panel.head(8))


## Multicollinearity Check

In [ ]:
X_cols = ['hdi', 'tpt', 'tpak', 'aps_1315', 'mean_years_schooling', 'expected_years_schooling', 'log_poverty_line']
X_vif = df[X_cols].copy()
X_vif_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data['Variable'] = X_cols
vif_data['VIF'] = [variance_inflation_factor(X_vif_const.values, i+1) for i in range(len(X_cols))]

print('Variance Inflation Factor (VIF)')
print('VIF > 10 indicates serious multicollinearity')
print(vif_data.sort_values('VIF', ascending=False).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#C00000' if v > 10 else '#FF7F00' if v > 5 else '#4472C4'
          for v in vif_data['VIF']]
bars = ax.barh(vif_data['Variable'], vif_data['VIF'], color=colors, alpha=0.85, edgecolor='black')
ax.axvline(5, color='orange', linestyle='--', alpha=0.8, label='VIF = 5 (moderate)')
ax.axvline(10, color='red', linestyle='--', alpha=0.8, label='VIF = 10 (serius)')
for bar, val in zip(bars, vif_data['VIF']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=10)
ax.set_xlabel('VIF')
ax.set_title('Variance Inflation Factor (VIF) per Variable')
ax.legend()
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plot_panel_vif.png', bbox_inches='tight')
plt.show()

high_vif = vif_data[vif_data['VIF'] > 10]['Variable'].tolist()
if high_vif:
    print(f'\nVariable dengan VIF > 10: {high_vif}')
    print('Pertimbanpoverty_linean untuk mengeluarkan atau menggabunpoverty_linean variable tersebut.')
else:
    print('\nAll variables are below the critical VIF threshold.')


In [ ]:

FORMULA_FULL = 'poverty_rate ~ ipm + tpt + tpak + aps_1315 + rls + hls + log_poverty_line'

FORMULA_PARSI = 'poverty_rate ~ ipm + tpt + tpak + log_poverty_line'

print('Model to be estimated:')
print(f'  Model Lenpoverty_lineap : {FORMULA_FULL}')
print(f'  Model Parsimoni: {FORMULA_PARSI}')


## Panel Model Estimation

The notebook estimates Pooled OLS, entity Fixed Effects, two-way Fixed Effects, and Random Effects models before selecting the final specification.

In [ ]:
exog_vars = ['hdi', 'tpt', 'tpak', 'aps_1315', 'mean_years_schooling', 'expected_years_schooling', 'log_poverty_line']

exog = sm.add_constant(df_panel[exog_vars])
endog = df_panel['poverty_rate']

mod_pooled = PooledOLS(endog, exog)
res_pooled = mod_pooled.fit(cov_type='robust')

print('POOLED OLS (Robust SE)')
print(res_pooled.summary)


In [ ]:
mod_fe = PanelOLS(endog, exog, entity_effects=True, time_effects=False)
res_fe = mod_fe.fit(cov_type='robust')

print('FIXED EFFECTS - Entity Effects (Robust SE)')
print(res_fe.summary)


In [ ]:
mod_fe2 = PanelOLS(endog, exog, entity_effects=True, time_effects=True)
res_fe2 = mod_fe2.fit(cov_type='robust')

print('FIXED EFFECTS - Two-Way (Entity + Time, Robust SE)')
print(res_fe2.summary)


In [ ]:
mod_re = RandomEffects(endog, exog)
res_re = mod_re.fit(cov_type='robust')

print('RANDOM EFFECTS (Robust SE)')
print(res_re.summary)


## Hausman Test: Fixed Effects vs Random Effects

The Hausman test evaluates whether Random Effects is consistent or whether Fixed Effects is preferred due to correlation between individual effects and regressors.

In [ ]:

common_vars = [v for v in exog_vars if v in res_fe.params.index and v in res_re.params.index]

b_fe = res_fe.params[common_vars]
b_re = res_re.params[common_vars]
diff = b_fe - b_re

V_fe = res_fe.cov.loc[common_vars, common_vars]
V_re = res_re.cov.loc[common_vars, common_vars]
V_diff = V_fe - V_re

try:
    V_diff_inv = np.linalg.pinv(V_diff.values)
    H_stat = float(diff.values @ V_diff_inv @ diff.values)
    df_h = len(common_vars)
    p_hausman = 1 - stats.chi2.cdf(H_stat, df_h)

    print('HAUSMAN TEST')
    print(f'H statistic   : {H_stat:.4f}')
    print(f'Degrees of freedom: {df_h}')
    print(f'p-value       : {p_hausman:.4f}')
    print()
    if p_hausman < 0.05:
        print('Decision: REJECT H0 (p < 0.05)')
        print('Rekomendasi: Gunakan FIXED EFFECTS')
        print('Interpretation: There is correlation between individual effects and regressors.')
    else:
        print('Decision: FAIL TO REJECT H0 (p >= 0.05)')
        print('Rekomendasi: Gunakan RANDOM EFFECTS (lebih efisien)')
        print('Interpretation: There is no significant correlation between individual effects and regressors.')
except Exception as e:
    print(f'Hausman test error: {e}')
    print('Use theoretical judgment: Fixed Effects is generally more conservative for provincial panel data.')


In [ ]:
print('INDIVIDUAL EFFECTS F-TEST (FE vs Pooled OLS)')
print('H0: There is no efek individu (Pooled OLS is sufficient)')
print('H1: There is efek individu significant (Fixed Effects is required)')
print()

print(f'F-statistic hasil FE model: {res_fe.f_statistic.stat:.4f}')
print(f'p-value F-stat: {res_fe.f_statistic.pval:.6f}')
if res_fe.f_statistic.pval < 0.05:
    print('Decision: REJECT H0: Fixed Effects significant diperlukan.')
else:
    print('Decision: FAIL TO REJECT H0: Pooled OLS munpoverty_linein mencukupi.')


## Fixed Effects Interpretation

In [ ]:
y_hat_fe = res_fe.fitted_values
residuals_fe = res_fe.resids

df_work = df_panel.copy()
df_work['fitted'] = y_hat_fe
df_work['resid'] = residuals_fe

prov_effects = df_work.groupby(level=0)['resid'].mean().sort_values()

print('Individual Fixed Effects by Province')
print('Positive values indicate provinces with poverty rates above the model prediction')
print('Negative values indicate provinces with poverty rates below the model prediction')
print()
print(prov_effects.round(4).to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
colors = ['#C00000' if v > 0 else '#4472C4' for v in prov_effects.values]
bars = ax.barh(range(len(prov_effects)), prov_effects.values, color=colors, alpha=0.8, edgecolor='black', linewidth=0.4)
ax.set_yticks(range(len(prov_effects)))
ax.set_yticklabels(prov_effects.index, fontsize=8)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Efek Tetap (Deviasi dari Average)')
ax.set_title('Individual Fixed Effects by Province\n(average residual FE model)')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plot_panel_fe_effects.png', bbox_inches='tight')
plt.show()


In [ ]:
models = {
    'Pooled OLS': res_pooled,
    'Fixed Effects': res_fe,
    'FE Two-Way': res_fe2,
    'Random Effects': res_re
}

print('MODEL COEFFICIENT COMPARISON')
print(f'{"Variable":15s}', end='')
for name in models:
    print(f'{name:>18s}', end='')
print()
print('-' * (15 + 18 * len(models)))

for var in exog_vars + ['const']:
    print(f'{var:15s}', end='')
    for name, res in models.items():
        if var in res.params:
            coef = res.params[var]
            pval = res.pvalues[var]
            stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
            print(f'{coef:>12.4f}{stars:>6s}', end='')
        else:
            print(f'{"N/A":>18s}', end='')
    print()

print()
print('Significantsi: *** p<0.01, ** p<0.05, * p<0.1')
print()

print('GOODNESS OF FIT')
print(f'{"Metrik":20s}', end='')
for name in models:
    print(f'{name:>18s}', end='')
print()
print('-' * (20 + 18 * len(models)))

metrics = {
    'R-squared': lambda r: getattr(r, 'rsquared', getattr(r, 'rsquared_between', np.nan)),
    'R-sq Within': lambda r: getattr(r, 'rsquared_within', np.nan),
    'R-sq Between': lambda r: getattr(r, 'rsquared_between', np.nan),
}

for metric_name, metric_fn in metrics.items():
    print(f'{metric_name:20s}', end='')
    for res in models.values():
        try:
            val = metric_fn(res)
            print(f'{val:>18.4f}', end='')
        except:
            print(f'{"N/A":>18s}', end='')
    print()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

model_pairs = [
    ('Fixed Effects', res_fe, axes[0]),
    ('Random Effects', res_re, axes[1]),
]

for model_name, res, ax in model_pairs:
    vars_plot = [v for v in exog_vars if v in res.params.index]
    coefs = res.params[vars_plot]
    se = res.std_errors[vars_plot]
    ci_lo = coefs - 1.96 * se
    ci_hi = coefs + 1.96 * se
    pvals = res.pvalues[vars_plot]

    colors_coef = ['#C00000' if p < 0.01 else '#FF7F00' if p < 0.05 else '#FFC000' if p < 0.1 else '#D3D3D3'
                   for p in pvals]

    y_pos = range(len(vars_plot))
    ax.barh(y_pos, coefs.values, color=colors_coef, alpha=0.8, edgecolor='black', linewidth=0.4)
    ax.errorbar(coefs.values, y_pos, xerr=1.96 * se.values,
                fmt='none', color='black', capsize=4, linewidth=1.2)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(vars_plot)
    ax.axvline(0, color='black', linewidth=1.2)
    ax.set_xlabel('Coefficients')
    ax.set_title(f'{model_name}\nCoefficients and 95% CI')
    ax.grid(axis='x', alpha=0.4)

    for i, (c, p) in enumerate(zip(coefs.values, pvals)):
        stars = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
        ax.text(c + (ci_hi.values[i] - c) + 0.05, i, stars, va='center', fontsize=9)

legend_patches = [
    plt.Rectangle((0,0), 1, 1, color='#C00000', alpha=0.8, label='p < 0.01 (***) significant sangat kuat'),
    plt.Rectangle((0,0), 1, 1, color='#FF7F00', alpha=0.8, label='p < 0.05 (**) significant kuat'),
    plt.Rectangle((0,0), 1, 1, color='#FFC000', alpha=0.8, label='p < 0.10 (*) significant lemah'),
    plt.Rectangle((0,0), 1, 1, color='#D3D3D3', alpha=0.8, label='p >= 0.10 not significant'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=2, fontsize=8.5,
           bbox_to_anchor=(0.5, -0.07))

plt.suptitle('Panel Regression Model Coefficient Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plot_panel_coefs.png', bbox_inches='tight')
plt.show()


## Residual Diagnostics

In [ ]:
resid_fe = res_fe.resids.values
fitted_fe = res_fe.fitted_values.values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.scatter(fitted_fe, resid_fe, alpha=0.6, s=50, color='steelblue', edgecolors='black', linewidths=0.3)
ax.axhline(0, color='red', linewidth=1.5, linestyle='--')
ax.set_xlabel('Fitted Values')
ax.set_ylabel('Residual')
ax.set_title('Residual vs Fitted Values')
ax.grid(alpha=0.4)

ax = axes[0, 1]
stats.probplot(resid_fe, dist='norm', plot=ax)
ax.set_title('Q-Q Plot Residual')
ax.grid(alpha=0.4)

ax = axes[1, 0]
ax.hist(resid_fe, bins=20, color='steelblue', edgecolor='black', alpha=0.8, density=True)
xmin, xmax = ax.get_xlim()
x_norm = np.linspace(xmin, xmax, 100)
p_norm = stats.norm.pdf(x_norm, resid_fe.mean(), resid_fe.std())
ax.plot(x_norm, p_norm, 'r-', linewidth=2, label='Normal fit')
ax.set_xlabel('Residual')
ax.set_ylabel('Density')
ax.set_title('Distribution Residual')
ax.legend()
ax.grid(alpha=0.4)

ax = axes[1, 1]
year_idx = df_panel.index.get_level_values(1)
for year in sorted(year_idx.unique()):
    mask_t = year_idx == year
    ax.scatter([year] * mask_t.sum(), resid_fe[mask_t],
               alpha=0.6, s=40, edgecolors='black', linewidths=0.3)
ax.axhline(0, color='red', linewidth=1.5, linestyle='--')
ax.set_xlabel('Year')
ax.set_ylabel('Residual')
ax.set_title('Residual by Year')
ax.set_xticks(sorted(year_idx.unique()))
ax.grid(alpha=0.4)

plt.suptitle('Residual Diagnostics: Fixed Effects Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plot_panel_diagnostics.png', bbox_inches='tight')
plt.show()

stat_sw, p_sw = stats.shapiro(resid_fe)
dw_stat = durbin_watson(resid_fe)

print('RESIDUAL DIAGNOSTIC TESTS')
print(f'Shapiro-Wilk test  : W = {stat_sw:.4f}, p = {p_sw:.4f}')
print(f'  Interpretation: {"Residuals are normally distributed" if p_sw > 0.05 else "Residuals are not normally distributed"}')
print(f'Durbin-Watson stat : {dw_stat:.4f}')
print(f'  Interpretation: {"No autocorrelation" if 1.5 < dw_stat < 2.5 else "Autocorrelation requires further investigation"}')
print(f'Mean residual      : {resid_fe.mean():.6f}')
print(f'Std residual       : {resid_fe.std():.4f}')


## Save Output Files

In [ ]:
coef_rows = []
for var in res_fe.params.index:
    coef_rows.append({
        'variable': var,
        'coefficient': res_fe.params[var],
        'std_error': res_fe.std_errors[var],
        't_stat': res_fe.tstats[var],
        'p_value': res_fe.pvalues[var],
        'ci_lower': res_fe.params[var] - 1.96 * res_fe.std_errors[var],
        'ci_upper': res_fe.params[var] + 1.96 * res_fe.std_errors[var],
        'significant': res_fe.pvalues[var] < 0.05
    })

df_coef = pd.DataFrame(coef_rows)
df_coef.to_csv(OUTPUT_DIR / 'output_fe_coefficient.csv', index=False)
print('Saved: output_fe_coefficient.csv')
print(df_coef.round(4).to_string(index=False))
print()

df_fitted = df_panel[['poverty_rate']].copy()
df_fitted['fitted_pooled'] = res_pooled.fitted_values
df_fitted['fitted_fe'] = res_fe.fitted_values
df_fitted['fitted_re'] = res_re.fitted_values
df_fitted['resid_fe'] = res_fe.resids
df_fitted = df_fitted.reset_index()

df_fitted.to_csv(OUTPUT_DIR / 'output_fe_fitted_residuals.csv', index=False)
print('Saved: output_fe_fitted_residuals.csv')
print(df_fitted.head(10).round(4).to_string(index=False))
print()

df_effects = prov_effects.reset_index()
df_effects.columns = ['province', 'fixed_effect']
df_effects = df_effects.sort_values('fixed_effect', ascending=False)
df_effects.to_csv(OUTPUT_DIR / 'output_fe_individual_effects.csv', index=False)
print('Saved: output_fe_individual_effects.csv')
print(df_effects.round(4).to_string(index=False))


In [ ]:
print('FIXED EFFECT PANEL REGRESSION SUMMARY')
print()
print(f'Selected model   : Fixed Effects (Entity)')
print(f'Number of provinces  : {df_panel.index.get_level_values(0).nunique()}')
print(f'Period          : {sorted(df_panel.index.get_level_values(1).unique())}')
print(f'Total observations  : {len(df_panel)}')
print()
print('Significant coefficients (p<0.05):')
sig_vars = df_coef[df_coef['significant']]
for _, r in sig_vars.iterrows():
    direction = 'positive' if r['coefficient'] > 0 else 'negative'
    print(f"  {r['variable']:15s}: coef = {r['coefficient']:.4f} ({direction}), p = {r['p_value']:.4f}")
print()
print('CSV Outputs:')
print('  1. output_fe_coefficient.csv          -> Fixed effects coefficients')
print('  2. output_fe_fitted_residuals.csv   -> Fitted values and residuals')
print('  3. output_fe_individual_effects.csv -> Fixed effects by province')
